# SpamGuard — Temel Python Projesi

Yaser Durdu için çalışma dosyası. İngilizce SMS üzerinde spam/normal sınıflandırması. Veri: Almeida & Hidalgo (2011), UCI SMS Spam Collection, https://doi.org/10.24432/C5CC84, CC BY 4.0.

Bu defter gerçek veriyle çalıştırılmıştır. Colab’a yükleyip Çalışma zamanı > Tümünü çalıştır ile yeniden üretilebilir. Son input hücresi kullanıcı girişi bekler.

## Dersteki temel Python ile bağlantı
14 Eylül kaydında Colab, açıklama hücreleri, metin işlemleri, input() ve f-string anlatılıyor. Bu çalışmada önce bu temel işlemleri görünür kılıyoruz. Veri setindeki gerçek spam/normal etiketinden model öğrenir. Aşağıdaki metin özelliklerinin hiçbiri tek başına spam kuralı değildir.

Colab kullanımı: Defteri yükleyin, üstten alta çalıştırın. İlk çalıştırmada internet bağlantısı ile UCI verisi indirilir. Yerel kurulum gerekmez. Son hücrede İngilizce bir mesaj girin. Kayıt inceleme notları: reports/Ders_Kaydi_01_Inceleme.md.

In [ ]:
# Değişken, string ve temel metin işlemleri: küçük gösterim.
mesaj_ornegi = "Free prize!!! Call now"
kelimeler = mesaj_ornegi.split()
karakter_sayisi = len(mesaj_ornegi)
unlem_sayisi = mesaj_ornegi.count("!")
kucuk_harfli_metin = mesaj_ornegi.lower()
print(f"Metin türü: {type(mesaj_ornegi).__name__}")
print(f"Kelime sayısı: {len(kelimeler)}")
print(f"Karakter sayısı: {karakter_sayisi}")
print(f"Ünlem sayısı: {unlem_sayisi}")
print(f"Küçük harfle: {kucuk_harfli_metin}")
# Bu bir tahmin değildir. Model için girdi özelliği üretmenin örneğidir.


## Proje Tanım Formu ve Plan
- **Problem cümlesi:** Bir İngilizce SMS mesajının uzunluk, rakam, ünlem, büyük harf ve bağlantı özelliklerinden spam olup olmadığını tahmin et.
- **Özellikler (X):** character_count, word_count, digit_count, exclamation_count, uppercase_ratio, url_count.
- **Hedef (y):** target; normal=0, spam=1. label bunun metin karşılığıdır. id yalnızca kayıt numarasıdır ve modele verilmez.
- **Problem türü:** İki sınıflı sınıflandırma.
- **Veri toplama yolu:** Hazır, etiketli UCI veri seti; anket veya API anahtarı kullanılmaz.

**Kime ve neden faydalı?** Çok sayıda İngilizce SMS alan kullanıcıların istenmeyen mesajları incelemesine yardımcı olabilecek bir eğitim prototipidir. Gereksiz bildirimleri ayırmayı hedefler. Yanlış tahmin olabileceği için otomatik mesaj silmek yerine kullanıcıya inceleme desteği sunar.

**Plan:** Veriyi oku → kalite kontrolü ve tekrar temizliği → altı özellik üret → veriyi ayır → EDA → karar ağacını eğit → test ve yeni örnekler → input ile mini araç. Eksik değer bulunmadığı için fillna veya dropna ile gereksiz değişiklik yapılmaz.

## Problem, veri ve hazırlık
Amaç, İngilizce SMS içeriğinden spam etiketini tahmin etmektir. Etiketi kaynak veri sağlar. Etiket = spam/ham; sayısal hedef = 1/0. Kimlik veya dosya adı özellik değildir. Tekrarları bölmeden önce çıkarırız. Eğitim %60, doğrulama %20, test %20. Test bölümünü model ayarı için kullanmayız.

### Kütüphaneler
Pandas tablo işlemleri, Matplotlib grafikler ve scikit-learn modelleme için kullanılır.

In [ ]:
from pathlib import Path
import io, zipfile, urllib.request, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier

### Veriyi oku
Yerelde dosya varsa onu kullanırız. Colab ortamında UCI arşivini indiririz. Bu kod ayrı bir kullanıcı hesabına veya API anahtarına ihtiyaç duymaz.

In [ ]:
# Colab: ham veri yoksa resmi UCI adresinden indir.
raw_path = Path('SMSSpamCollection')
if not raw_path.exists():
    local = Path('../data/SMSSpamCollection')
    if local.exists(): raw_path = local
    else:
        url='https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip'
        with urllib.request.urlopen(url, timeout=60) as response:
            archive=zipfile.ZipFile(io.BytesIO(response.read()))
        raw_path.write_bytes(archive.read('SMSSpamCollection'))
raw=pd.read_csv(raw_path,sep='\t',names=['label','text'],quoting=3,keep_default_na=False)

### Veriyi temizle
Boş/geçersiz kayıtları ve çelişen etiketleri kontrol ederiz. Küçük harf ve boşluk normalizasyonuyla tam tekrarları çıkarırız.

In [ ]:
raw['key']=raw.text.map(lambda s: re.sub(r'\s+',' ',s.strip().lower()))
conflicts=raw.groupby('key').label.nunique()
conflicts=set(conflicts[conflicts>1].index)
df=raw[(raw.key!='') & raw.label.isin(['ham','spam']) & ~raw.key.isin(conflicts)].drop_duplicates('key').copy().reset_index(drop=True)
df['target']=(df.label=='spam').astype(int)
df['id']=np.arange(len(df))
print('Eksik değerler:', raw.isnull().sum().to_dict())
print('Boş veya geçersiz kayıt:', int(((raw.key=='') | ~raw.label.isin(['ham','spam'])).sum()))
print('Çelişen etiketli metin:', len(conflicts))
print('Çıkarılan tekrar:', len(raw)-len(df))

### Altı özelliği üret
Her mesaj için aynı altı sayıyı hesaplarız. features() bir fonksiyondur. Harf sayısı sıfır olduğunda oran hesabını güvenli tutarız. Sütun isimleri FEATURES listesinde tutulur.

In [ ]:
def features(text):
    letters=[c for c in text if c.isalpha()]
    return [len(text),len(text.split()),sum(c.isdigit() for c in text),text.count('!'),sum(c.isupper() for c in letters)/max(len(letters),1),len(re.findall(r'https?://|www\.',text.lower()))]
FEATURES=['character_count','word_count','digit_count','exclamation_count','uppercase_ratio','url_count']
df[FEATURES]=pd.DataFrame(df.text.map(features).tolist(),index=df.index)

### Veriyi ayır
Özellikler X, öğrenilecek sınıf y olarak kullanılır. Aynı mesajın eğitim ve testte bulunmaması gerekir. Stratify sınıf oranlarını korur. Ayrıntılı parametre karşılaştırması ayrı ML çalışmasındadır.

In [ ]:
train,rest=train_test_split(df.index,test_size=.4,random_state=42,stratify=df.target)
val,test=train_test_split(rest,test_size=.5,random_state=42,stratify=df.loc[rest,'target'])
df['split']='train'
df.loc[val,'split']='validation'
df.loc[test,'split']='test'
df[['label','text','target']+FEATURES+['id','split']].to_csv('sms_features.csv',index=False)

### Kontrol çıktılarını incele
head(), describe() ve groupby sonuçlarını yalnızca üretmekle kalmayıp anlamlarını da yorumlayın. Aşağıda ham/temiz boyut, etiketler ve bölünme gösterilir.

In [ ]:
print('Ham tablo shape:', raw[['label','text']].shape)
print('Temiz tablo shape:', df.shape)
print('Ham/temiz:',len(raw),len(df))
print('Sınıf sayıları:',df.label.value_counts().to_dict())
print('Bölüm sayıları:',df.split.value_counts().to_dict())
print('Eksik değer:',df.isna().sum().sum())
print(df[['text','label']].head().to_string(index=False))

## Keşif
Eksik değer kontrolü, describe, groupby ve üç etiketli grafik. Normal mesajlar çoğunlukta olduğundan yalnızca doğrulukla karar vermeyiz. Özellik birimleri veri sözlüğünde açıklanır.

In [ ]:
print(df[FEATURES].describe().round(3).to_string())
print(df.groupby('label')[FEATURES].mean().round(3).to_string())
fig,axes=plt.subplots(1,3,figsize=(15,4))
df.label.value_counts().reindex(['ham','spam']).plot.bar(ax=axes[0],color=['#168a78','#e08245'],rot=0)
axes[0].set(title='Sınıf dağılımı',xlabel='Sınıf',ylabel='Mesaj sayısı')
for label in ['ham','spam']:
    axes[1].hist(df.loc[df.label==label,'character_count'],bins=np.arange(0,1000,25),alpha=.5,label=label,density=True)
axes[1].legend()
axes[1].set(title='Mesaj uzunluğu',xlabel='Karakter',ylabel='Yoğunluk')
df.groupby('label')[['digit_count','exclamation_count','url_count']].mean().T.plot.bar(ax=axes[2],rot=0,color=['#168a78','#e08245'])
axes[2].set(title='Ortalama özellikler',xlabel='Özellik',ylabel='Ortalama')
plt.tight_layout()
plt.savefig('veri_analizi.png',dpi=140)


### Keşif Bulguları
1. 5.159 mesajın 4.517’si normal, 642’si spamdır; dengesizlik nedeniyle doğruluk tek başına yeterli değildir.
2. Spam mesajları ortalama 137,45, normal mesajlar 70,90 karakterdir; uzunluk ayırıcı bilgi taşıyabilir, ancak dağılımlar örtüşür.
3. Spamda ortalama rakam sayısı 15,44, normalde 0,30’dur. Rakam sayısı aday bir ayırıcı özelliktir.
4. Spamda ünlem ortalaması 0,69, normalde 0,18; URL örüntüsü ortalaması sırasıyla 0,156 ve 0,0002’dir.
5. Bunlar veri seti içindeki ilişkilerdir; tek bir kelime, rakam veya bağlantı spam kanıtı değildir. Grafikler nedensellik göstermez.

## Model ve Seçim Gerekçesi
Hedef sürekli bir sayı değil, spam/normal sınıfıdır; bu nedenle DecisionTreeClassifier kullanılır. Karar ağacı, altı sayısal özelliği anlaşılır eşiklerle ayırır ve ölçekleme gerektirmez. Aşırı öğrenmeyi sınırlamak için min_samples_leaf=5 kullanılır; derinlik yalnızca doğrulama F1 ile seçilir, test verisi ayar için kullanılmaz.

In [ ]:
X=df[FEATURES]; y=df.target
candidates=[]
for depth in [3,5,8]:
    model=DecisionTreeClassifier(max_depth=depth,min_samples_leaf=5,class_weight='balanced',random_state=42)
    model.fit(X.loc[train],y.loc[train])
    score=f1_score(y.loc[val],model.predict(X.loc[val]))
    candidates.append((score,depth,model))
_,depth,tree=max(candidates,key=lambda x:x[0])
pred=tree.predict(X.loc[test])
print('Seçilen derinlik:',depth)
print('model.score doğruluk:', round(tree.score(X.loc[test],y.loc[test]),6))
print('Eğitim/test şekli:',X.loc[train].shape,X.loc[test].shape)
for name,fn in [('Accuracy',accuracy_score),('Precision',precision_score),('Recall',recall_score),('F1',f1_score)]:
    print(name,round(fn(y.loc[test],pred),6))
ConfusionMatrixDisplay(confusion_matrix(y.loc[test],pred),display_labels=['ham','spam']).plot(colorbar=False)
plt.title('Temel Python modeli')
plt.tight_layout()


### Değerlendirme Notu ve Aşırı Öğrenme
Aynı seçilmiş modelin eğitim, doğrulama ve test sonuçlarını karşılaştırırız. Test sonucuna bakarak parametreleri tekrar değiştirmeyiz.

In [ ]:
rows=[]
for name,indices in [('Eğitim',train),('Doğrulama',val),('Test',test)]:
    predictions=tree.predict(X.loc[indices])
    rows.append({'Bölüm':name,'Örnek':len(indices),'Doğruluk':accuracy_score(y.loc[indices],predictions),'Spam F1':f1_score(y.loc[indices],predictions)})
comparison=pd.DataFrame(rows)
print(comparison.round(4).to_string(index=False))
comparison.to_csv('python_bolum_karsilastirma.csv',index=False)


**Yorum:** Eğitim F1 %89.14, doğrulama F1 %90.71, test F1 %86.96. Bu karşılaştırma tek başına modelin dış dünyada güvenilir olduğunu kanıtlamaz. Derinlik 3 ile sınırlandı ve en az yaprak örneği 5 seçildi; bu önlemler aşırı öğrenme riskini azaltır, ortadan kaldırmaz. Aynı veri kaynağındaki benzer şablonlar sonucu iyimser gösterebilir. Güncel ve bağımsız SMS verisiyle dış test ve kelime özellikleri olası iyileştirmelerdir.

## Eğitimde bulunmayan üç yeni örnek
Bu cümleler gösterim için oluşturuldu. Resmî etiketli test başarısına eklenmedi.

In [ ]:
examples=['Can we meet at the library at six this evening?','Congratulations! Claim your free prize now. Call 09001234567!','Your order will arrive tomorrow. Thank you for shopping with us.']
example_features=pd.DataFrame([features(t) for t in examples],columns=FEATURES)
example_result=pd.DataFrame({'text':examples,'prediction':np.where(tree.predict(example_features)==1,'spam','ham')})
print(example_result.to_string(index=False))
example_result.to_csv('uc_yeni_ornek.csv',index=False)


### Yeni Örneklerin Yorumu
- Buluşma mesajı **normal** tahmin edildi; gündelik bir iletişim örneğidir.
- Ücretsiz ödül ve telefon numarası içeren mesaj **spam** tahmin edildi; çok sayıda rakam ve ünlem gibi özellikler içerir.
- Sipariş bildirimi **normal** tahmin edildi; olağan bir bilgilendirme örneğidir.

Bu üç cümle gösterim için yazılmıştır; bağımsız gerçek etiketleri yoktur. Modelin yalnızca altı sayısal özellik kullandığı unutulmamalıdır; cümlelerin anlamını tam olarak kavradığı sonucuna varamayız. Bunları test başarısına katmadık.

## Mini uygulama
Bu son hücre kullanıcıdan girdi bekler. Yüklü model İngilizce SMS için geliştirilmiştir.

In [ ]:
def tahmin_et(text):
    if not text.strip(): raise ValueError('Boş mesaj girilemez.')
    row=pd.DataFrame([features(text)],columns=FEATURES)
    return 'spam' if tree.predict(row)[0] else 'normal'

mesaj=input('İngilizce SMS girin: ')
print(f'Modelin tahmini: {tahmin_et(mesaj)}')


## Sonuç
Test doğruluğu %96,51, spam F1 %86,96. Yüksek recall ile spamların çoğunu yakalarken precision düşüyor. Daha güçlü bir yaklaşım metnin kelime örüntülerini kullanabilir. Etiketli dış test, yeni tarihli veri ve Türkçe veri ile ayrı doğrulama gerekir.